# Module 5 · Analysis, Visualisation & Narrative

*Jupyter Book, notebook 3 of 3: M2 obtain > M4 filter > **M5 analyse.***

Now we answer the research question with a simple count and chart, and write a short, honest narrative. Keep conclusions modest, describe the distribution, don't over-interpret.

**No API key needed**, uses the filtered file from Module 4.


## Settings


In [1]:
from pathlib import Path
import pandas as pd

# Candidate locations for the shared datasets/ folder (repo root, or one level up from a module).
_DATASET_NAMES = ["van_gogh_combined.csv", "van_gogh_europeana.csv", "van_gogh_filtered.csv"]

def find_dataset(name):
    """Return the path to a dataset file, wherever the notebook is run from."""
    for base in [Path("datasets"), Path("../datasets")]:
        if (base / name).exists():
            return base / name
    raise FileNotFoundError(f"{name} not found in datasets/, run the earlier notebook first.")

def datasets_dir():
    """The datasets/ folder that actually holds the data (so saves land beside the real files)."""
    for base in [Path("datasets"), Path("../datasets")]:
        if any((base / n).exists() for n in _DATASET_NAMES):
            return base
    Path("datasets").mkdir(parents=True, exist_ok=True)   # first run, live mode
    return Path("datasets")

import matplotlib.pyplot as plt

GROUP_BY = "institution"     # try "country", "type", "year"
RESEARCH_QUESTION = "How are Van Gogh's works distributed across the participating institutions?"
PROC = Path("data/processed"); PROC.mkdir(parents=True, exist_ok=True)

## 1 · Load the filtered data (and a word on cleaning)

Cleaning here means small, honest tidying, consistent text, no blank groups, *before* counting. We do **not** invent or change values.


In [2]:
def load_filtered():
    for name in ["van_gogh_filtered.csv", "van_gogh_combined.csv", "van_gogh_europeana.csv"]:
        try:
            p = find_dataset(name); print("Loaded", p); return pd.read_csv(p)
        except FileNotFoundError:
            continue
    raise FileNotFoundError("Run Module 4 first.")

df = load_filtered()
df[GROUP_BY] = df[GROUP_BY].astype(str).str.strip()      # trim stray spaces (light cleaning)
df = df[df[GROUP_BY] != ""]
print("Records:", len(df), "| grouping by:", GROUP_BY)

Loaded ../datasets/van_gogh_filtered.csv
Records: 34 | grouping by: institution


## 2 · Count and visualise


In [ ]:
counts = df[GROUP_BY].value_counts()
print(counts)

# Horizontal bars read better for long category labels (e.g. institution names).
ordered = counts.sort_values()                       # ascending, largest ends up on top
ax = ordered.plot(kind="barh", figsize=(9, 5), color="#2E75B6")
ax.set_title(RESEARCH_QUESTION, fontsize=11)
ax.set_xlabel("Number of works"); ax.set_ylabel(GROUP_BY.capitalize())
ax.set_xticks(range(0, int(ordered.max()) + 1, 2))   # whole-number ticks (0, 2, 4, ...)
plt.tight_layout()
plt.savefig(PROC / f"works_per_{GROUP_BY}.png", dpi=150)
plt.show()

In [ ]:
counts.rename("count").to_csv(PROC / f"works_per_{GROUP_BY}.csv")
print("Saved table + chart to", PROC)

## 3 · Narrative, describe what you see

Write 3-4 plain sentences. A helpful shape (fill in from *your* chart):

- **What the distribution looks like:** which institution/country holds the most; how concentrated or spread it is.
- **One thing that stands out:** e.g. a single institution holding a large share.
- **A limitation:** the data reflect what institutions catalogued on Europeana and how they wrote the creator's name, not the artist's full output. (See the name-variation lesson in Module 2.)
- **A next question** this raises.

> Keep it modest and evidence-based, the goal is a transparent description, not a grand claim.


### Example interpretation *(write your own first, then compare)*

> Of **34** catalogued works across **10 institutions**, the **Catholic University of Leuven** accounts
> for **18 (53%)**, not because it holds the most Van Goghs, but because its teaching collection
> uploads photographic study reproductions. The remaining 16 are spread thinly across nine institutions
> in eight countries (Belgium 18, Netherlands 8, then Austria, Sweden, and four countries with one each).
> The clearest pattern is **concentration in a single provider**. **Limitation:** this reflects which
> institutions catalogued works under Van Gogh's name on Europeana, and how, not where his paintings
> actually hang. **Next question:** does the picture change if we group by `country`, or if we include the
> reproduction index we set aside in Module 4?

This is *one* honest reading, modest, tied to the numbers, and explicit about its limits.


**Workflow note (for your README):** *Counted works per institution on the filtered dataset and produced a bar chart; wrote a short narrative noting the concentration in a few institutions and the coverage/cataloguing limitations.*

 That completes the obtain > filter > analyse arc on one shared dataset.
